# Prep output

## Paralleled index

In [ ]:
from sklearn.model_selection import KFold

import pandas as pd
import numpy as np
import datasets

import warnings
warnings.simplefilter(action='ignore')

save_data = False

In [ ]:
df = pd.read_parquet("hf://datasets/malmaud/onestop_qa/data/train-00000-of-00001.parquet")

indices = np.arange(len(df)//3//3)

# Sort the dataset by levels and separate
df = df.sort_values('level',kind='stable').reset_index(drop=True)
l1_df = df.loc[0:485]
l2_df = df.loc[486:486+485].reset_index(drop=True)
l3_df = df.loc[486+485+1:486+485+1+485].reset_index(drop=True)

kf = KFold(n_splits=9,random_state=42,shuffle=True)
kf.get_n_splits(indices)

import itertools
def pairwise(iterable):
    "s -> (s0, s1), (s1, s2), (s2, s3), ..., (s8, s0)"
    a, b = itertools.tee(iterable)
    next(b, None)
    return zip(a, b)

test_indices = []
for i, (train_index, test_index) in enumerate(kf.split(indices)):
    test_indices.append(test_index)

test_val_indices = list(pairwise(test_indices))
test_val_indices.append((test_indices[-1],test_indices[0]))

## Specifying B-type data

test_dfs = []
for i, test_idx in enumerate(test_indices):
    # Use only level 1 version of the passages for testing
    q_indices_test = sorted(np.concatenate((test_idx * 3, test_idx * 3 + 1, test_idx * 3 + 2)))
    test_df = l1_df.iloc[q_indices_test,:]
    test_df['correct'] = test_df['answers'].str[0]
    test_df = test_df.explode('answers').reset_index(drop=False)
    test_df['paragraph'] = test_df['paragraph'].str.split()
    test_df['option'] = ['A', 'B', 'C', 'D'] * len(q_indices_test)
    test_df['qa'] = test_df['question'] + ' ' + test_df['answers']
    test_df['qa'] = test_df['qa'].str.split()
    test_df = test_df.drop(axis='index', index=[item for item in test_df.index.to_list() if item % 4 == 0])
    
    # Select only B
    test_df_B = test_df[test_df['option'] == 'B'].reset_index(drop=True)
    test_dfs.append(test_df_B)

In [ ]:
l1_df.iloc[test_dfs[0]['index'].values].head(3)

In [ ]:
print(test_dfs[0].shape)
test_dfs[0].head(3)

In a word, the index column of `test_df` is parallel with the index of `onestop_qa`.

In [ ]:
pd.concat(test_dfs).shape

## Sorting output .json files

In [ ]:
all_df.merge(test_dfs[-1][['index','correct']], left_on='outer_index', right_on='index')['correct']

In [ ]:
import re
import json
from itertools import product

all_dfs = []
distractor_dfs = []

for file_idx in range(0,9):
    json_path = f'sft-b-{file_idx+1}.json'
    with open(json_path) as f:
        data = json.load(f)

    rows = []
    for outer_key, inner_dict in data.items():
        for inner_key, content in inner_dict.items():
            if not content:  
                content = {'value': None}  
            
            row = {
                'outer_index': int(outer_key),  # Serial index for the test_df
                'inner_index': int(inner_key),  # Index for the times of trials
                **content  
            }
            rows.append(row)

    # def rematch_distractor(text):
    #     preserved = r"a-zA-Z0-9\s,:\.?"  

    #     cleaned_punc = re.sub(
    #         pattern=f"[^{preserved}]", 
    #         repl="", 
    #         string=text,
    #         flags=re.UNICODE
    #     )
        
    #     cleaned_punc = re.sub(r"\s+", " ", cleaned_punc).strip()

    #     critical_match = re.search(r'(?i)(?:critical\s+)?span[^:]*:\s*(.*)', cleaned_punc, re.IGNORECASE)
    #     if critical_match:
    #         critical_span = critical_match.group(1).strip()
    #     else:
    #         critical_span = ''

    #     distractor_match = re.search(r'(?i)distractor[^:]*:\s*(.*?\.)(?=\s|$)', cleaned_punc, re.IGNORECASE)
    #     if distractor_match:
    #         distractor = distractor_match.group(1).strip()
    #     else:
    #         distractor = ''

    #     return critical_span, distractor

    def rematch_distractor(text):
        # cleaned_punc = re.sub(
        #     pattern=f"\\", 
        #     repl="", 
        #     string=text,
        #     flags=re.UNICODE
        # )
        
        # cleaned_punc = re.sub(r"\s+", " ", text).strip()
        cleaned_punc = text.strip()

        critical_match = re.search(r'(?i)(?:critical\s+)?span[^:]*:\s*(.*)', cleaned_punc, re.IGNORECASE)
        if critical_match:
            critical_span = critical_match.group(1).strip()
        else:
            critical_span = ''

        # distractor_match = re.search(r'(?i)distractor[^:]*:\s*(.*?\.)(?=\s|$)', cleaned_punc, re.IGNORECASE)
        distractor_match = re.search(r'(?i)distractor(?:\s+\w+)?:\s*(.*?(?:\.|\n|$))', cleaned_punc, re.IGNORECASE)    

        if distractor_match:
            distractor = distractor_match.group(1).strip()
        else:
            distractor = ''

        return critical_span, distractor

    all_df = pd.DataFrame(rows)

    all_df['rematch_span'] = all_df['output'].apply(lambda x: rematch_distractor(x)[0])
    all_df['rematch_distractor'] = all_df['output'].apply(lambda x: rematch_distractor(x)[1])
    all_df['outer_index'] = all_df['outer_index'].apply(lambda x: test_dfs[file_idx]['index'].values[x])
    all_df['model'] = f'fold-{file_idx+1}'
    all_df = all_df.drop('temperature', axis=1)
    all_df = all_df.loc[:,[
        'outer_index',
        'inner_index',
        'model',
        'output',
        'source_article',
        'source_question',
        # 'correct',
        'rematch_span',
        'rematch_distractor',
        'validity',
        'critical_span',
        'distractor'
    ]]

    all_dfs.append(all_df)


    # distractor_df = all_df[all_df['validity'] == 'F']
    # Some generated distractors may be constantly true
    distractor_df = all_df.groupby('outer_index').tail(1)
    distractor_df['correct'] = test_dfs[file_idx]['correct'].values
    distractor_df = distractor_df.loc[:,[
        'outer_index',
        'inner_index',
        'model',
        'output',
        'source_article',
        'source_question',
        'correct',
        'rematch_span',
        'rematch_distractor',
        'validity',
        'critical_span',
        'distractor'
    ]]

    distractor_dfs.append(distractor_df)

all_unfiltered_df = pd.concat(all_dfs)
print(all_unfiltered_df.shape)
all_distractor_df = pd.concat(distractor_dfs)
print(all_distractor_df.shape)

In [ ]:
all_distractor_df[all_distractor_df['validity'] == 'T']

In [ ]:
if save_data:
    all_unfiltered_df.to_excel(f"{all_unfiltered_df.shape[0]}_sft_B_all.xlsx")
    all_distractor_df.to_excel(f"{all_distractor_df.shape[0]}_sft_B_distractors.xlsx")